# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1 1.3333 
[1, None, 3] -> [False, True, False] -> 1
[1, None, 3] -> [1, 0, 3] -> 4/3 -> 1.3333
2. [" a", "A ", "b"] -> ["a", "A", "b"] -> ["a", "a", "b"] -> ['a', 'a'] -> 2
3. [1, 1, 2] -> [1, 2] -> 2
4. ["1 000", "5"] -> [1000, 5] -> 1005
5. [9, 10]
Меняем тип данных на дату и время, затем вычленяем номер месяца и получаем список.

In [82]:
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

1 1.3333333333333333
2
3
1005
[9, 10]


In [83]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [84]:
# 1
import pandas as pd


def clean(path: str) -> pd.DataFrame:
    CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
            "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
            "актобе": "Актобе", "атырау": "Атырау"}

    df = pd.read_csv(path)
    print(f'Шаг чтения файла, кол-во строк {len(df)}')
    df = df[(df['customer_id'] != 'TEST')]
    print(f'Шаг удаления тестовых клиентов, кол-во строк {len(df)}')
    df = df.drop_duplicates()
    print(f'Шаг удаления дублей, кол-во строк {len(df)}')
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print(f'Проверка NaN по полю city - {"Пропусков нет" if df["city"].isna().sum() == 0 else 'Есть пропуски'}')
    print(f'Проверка NaN по полю amount - {"Пропусков нет" if df["amount"].isna().sum() == 0 else 'Есть пропуски'}')
    print(f'Проверка уникальности нмоера заказа - {"Дублей нет" if df["order_id"].is_unique else "Есть дубли"}')

    return df


df = clean('orders.csv')

Шаг чтения файла, кол-во строк 5463
Шаг удаления тестовых клиентов, кол-во строк 5460
Шаг удаления дублей, кол-во строк 5280
Проверка NaN по полю city - Пропусков нет
Проверка NaN по полю amount - Пропусков нет
Проверка уникальности нмоера заказа - Дублей нет


Очистка на входе нужна чтобы избежать искажений в результате анализа.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [85]:
# 2
sep_amt = df[(df['order_date'].dt.month == 9) & (df['status'] == 'оплачен')]['amount'].sum()
oct_amt = df[(df['order_date'].dt.month == 10) & (df['status'] == 'оплачен')]['amount'].sum()

cities = df['city'].unique()

for city in cities:
    sep_amt_city = df[(df['order_date'].dt.month == 9) & (df['city'] == city) & (df['status'] == 'оплачен')]['amount'].sum()
    oct_amt_city = df[(df['order_date'].dt.month == 10) & (df['city'] == city) & (df['status'] == 'оплачен')]['amount'].sum()
    res = (sep_amt_city - oct_amt_city) / (sep_amt - oct_amt) * 100
    print(f"{city:<10} {res:>12,.4f}")

Алматы         -17.2292
Астана          22.2955
Караганда       33.6016
Шымкент         62.4646
Атырау          -5.5675
Актобе           4.4351


Совпадает ли с таблицей «город × месяц» урока 4?
Расчет вышел идентичным.
Какой город падает сильнее всех?
Шымкент

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [86]:
# 3
df = pd.read_csv('orders.csv')
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["order_date"] = pd.to_datetime(df["order_date"])

paid = df['status'] == 'оплачен'
sep = df['order_date'].dt.month == 9
oct = df['order_date'].dt.month == 10

sep_amt = df.loc[paid & sep, "amount"].sum()
oct_amt = df.loc[paid & oct, "amount"].sum()
print(f'Шаг чтения файла, сумма выручки {(oct_amt-sep_amt)/sep_amt:.1%}')

df = df[(df['customer_id'] != 'TEST')]
sep_amt = df.loc[paid & sep, "amount"].sum()
oct_amt = df.loc[paid & oct, "amount"].sum()
print(f'Шаг удаления ТЕСТ клиента, сумма выручки {(oct_amt-sep_amt)/sep_amt:.1%}')

df = df.drop_duplicates()
sep_amt = df.loc[paid & sep, "amount"].sum()
oct_amt = df.loc[paid & oct, "amount"].sum()
print(f'Шаг удаления дубликатов, сумма выручки {(oct_amt-sep_amt)/sep_amt:.1%}')


Шаг чтения файла, сумма выручки -22.7%
Шаг удаления ТЕСТ клиента, сумма выручки -8.2%
Шаг удаления дубликатов, сумма выручки -2.3%


Без удаления тестового клиента у нас -22.7% после удаления -8,2%. Сильнее искажает именно наличие тестового клиента.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [87]:
# 4
df = clean('orders.csv')
isna_mask = df['delivery_days'].isna()
print('Сколько пропусков в delivery_days по чистым данным? ', isna_mask.sum())
df.loc[isna_mask, ['city','status']].value_counts()

Шаг чтения файла, кол-во строк 5463
Шаг удаления тестовых клиентов, кол-во строк 5460
Шаг удаления дублей, кол-во строк 5280
Проверка NaN по полю city - Пропусков нет
Проверка NaN по полю amount - Пропусков нет
Проверка уникальности нмоера заказа - Дублей нет
Сколько пропусков в delivery_days по чистым данным?  158


city       status 
Алматы     оплачен    61
Астана     оплачен    35
Караганда  оплачен    23
Шымкент    оплачен    17
Астана     отменён     5
Шымкент    отменён     5
Актобе     оплачен     4
Атырау     оплачен     4
Алматы     отменён     2
Атырау     отменён     1
Караганда  отменён     1
Name: count, dtype: int64

Лучше удалить, потому что если заполнить медианой или 0 то произойдут искажения при вычислениях.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [88]:
# 5

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df = df.drop_duplicates()
df["city"] = df["city"].str.strip().str.lower().map(CITY)
print(df["city"].nunique(), len(df))

6 5283


Исправил удаление дубликатов и сделал мапинг городов, теперь все верно.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [89]:
# 6
df = clean('orders.csv')
df['day_name'] = df['order_date'].dt.day_name()
print(df['day_name'].value_counts(sort=True))
by_days = df['day_name'].value_counts(sort=True)
print('В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?')
print(f"{(by_days.iloc[-1] - by_days.iloc[0])/by_days.iloc[0]:.1%}")


Шаг чтения файла, кол-во строк 5463
Шаг удаления тестовых клиентов, кол-во строк 5460
Шаг удаления дублей, кол-во строк 5280
Проверка NaN по полю city - Пропусков нет
Проверка NaN по полю amount - Пропусков нет
Проверка уникальности нмоера заказа - Дублей нет
day_name
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
-13.0%


В понедельник нужно меньше курьеров. Компания может сэкономить на штате.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [90]:
# 7
df['month_name'] = df['order_date'].dt.month_name()

shymkent = (df['city'] == 'Шымкент')

oct_shym = df[(df['month_name'] == 'October') & shymkent]['amount'].mean()
sep_shym = df[(df['month_name'] == 'September') & shymkent]['amount'].mean()

first_date = df.loc[(df['delivery_days'] >= 5) & shymkent, 'order_date'].sort_values().iloc[0]

other_cities = len(df.loc[(df['delivery_days'] >= 5) & (df['order_date'] >= first_date) & ~shymkent, 'order_date'].sort_values())

print(f"Кол-во дат с таким же условием в других городах {other_cities}")

Кол-во дат с таким же условием в других городах 0


Взять отчет у отдела логистики, необходимо проверить кол-во курьеров в сентябре и октябре. Похоже текущий штат логистики не справляется с потоком доставок клиентам.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [106]:
# 8
def to_amount(s):
    return pd.to_numeric(s.str.strip().str.replace(' ', '').str.replace(',', '.'), errors="coerce")

In [108]:
test_series = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print(to_amount(test_series))



0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64


In [110]:
raw_df = pd.read_csv('orders.csv')
raw_df = raw_df.drop_duplicates()
raw_df = raw_df[(raw_df['customer_id'] != 'TEST')]
raw_df['amount'] = to_amount(raw_df['amount'])

print('Проверка сходства с эталоном', end=' - ')
print(raw_df['amount'].sum() == df['amount'].sum())
print('Проверка кол-ва пропусков по exception', end=' - ')
print(df["amount"].isna().sum())

Проверка сходства с эталоном - True
Проверка кол-ва пропусков по exception - 0


Такие функции важны чтобы придерживаться правила DRY. То есть они позволяют переиспользовать готовый код, что придает компактности и понятности конечному скрипту.